# 05b. 뉴스 보정: 수익률 예측에 뉴스 점수 더하기

05에서는 현재가 유지(Naive)에 뉴스 계수를 더해 봤고 채택하지 않았다. 03b에서 수익률 예측을 다시 설계했으므로, 이번에는 그 서비스 예측 위에 뉴스 보정을 더해 같은 규칙으로 한 번 더 시험한다. 먼저 03b 예측이 Naive와 얼마나 다른지(λ의 크기)를 본다.

## 사전 등록 (계산 전에 고정, 2026-10-05)

**자료**
- 보관 기사 1,446건(2022-01-04 ~ 2026-09-25)의 Jev 점수. 개발 기록 24의 판단에 따라 "발행 다음 날 알았다"는 연구용 시점(`daily_news(mode="research")`)을 쓴다. 기사가 없는 날의 점수는 0이다(05와 같다).
- 피처는 거래일 t 마감까지 알려진 뉴스 점수 s_t 하나다(05와 같은 정의).

**보정**
- 기준 예측: 03b 서비스 방식의 연도별 예측(그해 전 자료로 고른 설정 × λ).
- 평가 연도 Y마다 2022 … Y−1년 행(목표일이 Y−1년 말까지)에서 잔차 e = 실제 − 기준 예측을 s로 회귀해 γ = Σ(s·e) / Σ(s²)를 구한다(절편 없음). Y년 예측 = 기준 예측 + γ·s.
- 지평: 5일(주 시험, '짧은 지평은 뉴스에 민감하다'는 가설), 20일(보조).

**평가와 판단 규칙**
- 평가 연도: 2023, 2024, 2025, 2026년(목표일 가격이 확인된 행). 05에서 2023–2025년을 같은 방식으로 이미 시험했으므로 재사용 확인 구간이다. 진짜 평가는 live 점수다.
- 채택 규칙(05와 같다): 네 해를 합친 ΔRMSE(보정 − 기준)의 블록 bootstrap(블록 = h, 1,000회) 95% 구간 상한이 0보다 작고, 네 해 모두 RMSE가 줄어야 채택한다.
- 채택하면 서비스 예측에 보정 항을 넣고(서비스는 live 시점 점수), 아니면 넣지 않는다. 어느 쪽이든 live 점수로 같은 규칙의 재검증을 이어 간다.
- 결과를 본 뒤 피처 정의, 지평, 규칙을 바꾸지 않는다.

In [1]:
import numpy as np
import pandas as pd

import common
from coffee.config import HORIZONS
from coffee.evaluate import block_bootstrap_rmse_diff, fold_rows
from coffee.features import ALL_FEATURES, EXTRA_FEATURES, build_dataset
from coffee.models import return_model
from coffee.news import daily_news, load_jev_archive
from coffee.sources import load_sources

data = build_dataset(load_sources())
articles = load_jev_archive()
score = daily_news(articles, data.index, mode="research")["news_score"].fillna(0.0)  # 기사가 없는 날은 0
redesign = {int(h): v for h, v in common.load_result("return_redesign")["horizons"].items()}
ROWS = ALL_FEATURES + EXTRA_FEATURES  # 03b와 같은 행


def service_predictions(h, years):
    """03b 서비스 방식의 연도별 예측. 해마다 그 전까지의 자료로 고른 설정을 학습한다(행 번호 → 줄이기 전 예측 r̂, λ)."""
    y, parts = data[f"y_{h}"].to_numpy(), []
    for year in years:
        c = redesign[h]["by_year"][str(year)]
        fit, test = fold_rows(data, ROWS, h, year, 2026)
        if c["candidate"] == "Naive":
            raw, weight = np.zeros(len(test)), 0.0
        else:
            raw = return_model(c["model"], c["features"], h, c["params"], c["scale"]).fit(data, fit, y[fit]).predict(data, test)
            weight = c["weight"]
        parts.append(pd.DataFrame({"raw": raw, "weight": weight}, index=test))
    frame = pd.concat(parts)
    rows = frame.index.to_numpy()
    return frame.assign(pred=frame["weight"] * frame["raw"], y=data[f"y_{h}"].to_numpy()[rows],
                        target=data[f"target_date_{h}"].to_numpy()[rows], year=data.index[rows].year,
                        s=score.to_numpy()[rows])


base = {h: service_predictions(h, range(2012, 2027)) for h in HORIZONS}
print(f"기사 {len(articles):,}건, {articles['event_at'].min():%Y-%m-%d} ~ {articles['event_at'].max():%Y-%m-%d}")

기사 1,446건, 2022-01-04 ~ 2026-09-25


## 1. 03b 예측은 Naive와 얼마나 다른가

λ는 줄이기 전 예측 r̂에 곱하는 값이라 λ만 보면 예측이 얼마나 작은지 알 수 없다. 줄인 뒤 예측의 크기를 실제 움직임과 비교하고, 줄이기 전 예측과 실제의 상관 ρ를 본다. 예측을 가장 잘 줄였을 때 Naive보다 낮출 수 있는 RMSE는 √(1 − ρ²) − 1이 한계다.

In [2]:
PERIODS = {"개발 2012–2021": (2012, 2021), "재사용 확인 2022–2025": (2022, 2025), "2026년": (2026, 2026)}
rows = []
for h in HORIZONS:
    frame = base[h]
    for name, (first, last) in PERIODS.items():
        part = frame[(frame["year"] >= first) & (frame["year"] <= last) & (frame["target"] <= pd.Timestamp(f"{last}-12-31"))]
        used = part[part["weight"] > 0]
        rho = np.corrcoef(used["raw"], used["y"])[0, 1] if len(used) > 2 else np.nan
        rms_y = np.sqrt(np.mean(part["y"] ** 2))
        rows.append({"지평": h, "구간": name, "λ 평균": part["weight"].mean(), "Naive를 고른 행": 1 - len(used) / len(part),
                     "줄인 예측 크기 / 실제": np.sqrt(np.mean(part["pred"] ** 2)) / rms_y,
                     "평균 |예측| %": 100 * np.mean(np.abs(np.expm1(part["pred"]))),
                     "평균 |실제| %": 100 * np.mean(np.abs(np.expm1(part["y"]))),
                     "상관 ρ (줄이기 전)": rho, "ρ로 얻을 수 있는 최대 개선 %": 100 * (np.sqrt(1 - rho ** 2) - 1)})
shrink_table = pd.DataFrame(rows)
print({h: round(redesign[h]["final"]["weight"], 3) for h in HORIZONS}, "서비스 λ")
shrink_table.set_index(["지평", "구간"]).round(3)

{5: 0.292, 20: 0.214, 60: 0.072} 서비스 λ


λ 평균  Naive를 고른 행  줄인 예측 크기 / 실제  평균 |예측| %  평균 |실제| %  \
지평 구간                                                                          
5  개발 2012–2021      0.314        0.000          0.068      0.219      3.533   
   재사용 확인 2022–2025  0.166        0.000          0.049      0.160      4.056   
   2026년             0.292        0.000          0.058      0.249      4.640   
20 개발 2012–2021      0.288        0.000          0.156      0.939      6.646   
   재사용 확인 2022–2025  0.208        0.000          0.106      0.893      8.089   
   2026년             0.214        0.000          0.061      0.632     10.476   
60 개발 2012–2021      0.096        0.702          0.087      0.459     10.128   
   재사용 확인 2022–2025  0.027        0.308          0.017      0.161     12.823   
   2026년             0.072        0.000          0.027      0.339     13.543   

                     상관 ρ (줄이기 전)  ρ로 얻을 수 있는 최대 개선 %  
지평 구간                                                  
5  개발 2012–2021             0.025              -0.031  
   재사용 확인 2022–2025         0.066              -0.219  
   2026년                    0.003              -0.001  
20 개발 2012–2021             0.133              -0.884  
   재사용 확인 2022–2025         0.094              -0.442  
   2026년                    0.010              -0.005  
60 개발 2012–2021            -0.219              -2.425  
   재사용 확인 2022–2025         0.296              -4.487  
   2026년                    0.139              -0.971

서비스 λ는 5 / 20 / 60일 0.29 / 0.21 / 0.07이다. 줄인 뒤 예측의 크기는 실제 움직임의 5일 5–7%, 20일 6–16%, 60일 2–9%이고, 평균으로 보면 5일 0.2%, 20일 0.6–0.9%, 60일 0.2–0.5%만 움직인다고 예측한다(실제는 3.5–4.6%, 6.6–10.5%, 10.1–13.5%). 가격 수준으로는 Naive에 가깝다.

이유는 상관 ρ에 있다. 5일은 ρ가 0.00–0.07이라 예측을 가장 잘 줄여도 Naive보다 0.2% 넘게 낮출 수 없다. 20일은 개발 구간 ρ 0.13으로 0.9%까지 가능했고 실제로 0.53%를 얻었지만, 2026년 ρ는 0.01이다. 60일은 ρ가 −0.22(개발)에서 +0.30(재사용 확인)으로 부호까지 바뀌어, λ가 0 근처에 머문 것이 맞는 선택이다. λ가 작다는 것은 모델이 '현재가에서 크게 벗어날 근거가 없다'고 스스로 말하는 것이다.

## 2. 뉴스 보정 시험 (사전 등록한 규칙)

In [3]:
EVAL_YEARS = (2023, 2024, 2025, 2026)


def news_test(h):
    frame = base[h][base[h]["year"] >= 2022]
    rows, pooled = [], {"y": [], "base": [], "news": []}
    for year in EVAL_YEARS:
        fit = frame[(frame["year"] < year) & (frame["target"] <= pd.Timestamp(f"{year - 1}-12-31"))]
        test = frame[frame["year"] == year]
        gamma = float((fit["s"] * (fit["y"] - fit["pred"])).sum() / (fit["s"] ** 2).sum())
        corrected = test["pred"] + gamma * test["s"]
        rmse_base, rmse_news = np.sqrt(np.mean((test["y"] - test["pred"]) ** 2)), np.sqrt(np.mean((test["y"] - corrected) ** 2))
        rows.append({"지평": h, "연도": year, "학습 행": len(fit), "평가 행": len(test), "γ": gamma, "RMSE 기준": rmse_base,
                     "RMSE 뉴스 보정": rmse_news, "변화율(%)": 100 * (rmse_news / rmse_base - 1)})
        pooled["y"].append(test["y"].to_numpy())
        pooled["base"].append(test["pred"].to_numpy())
        pooled["news"].append(corrected.to_numpy())
    pooled = {key: np.concatenate(value) for key, value in pooled.items()}
    ci = block_bootstrap_rmse_diff(pooled["y"], pooled["news"], pooled["base"], block=h)
    adopted = bool(ci["high"] < 0 and all(row["RMSE 뉴스 보정"] < row["RMSE 기준"] for row in rows))
    return rows, ci, adopted


results = {h: news_test(h) for h in (5, 20)}
for h, (_, ci, adopted) in results.items():
    print(f"{h}거래일: ΔRMSE(보정 − 기준) = {ci['diff']:+.5f}, 95% 구간 [{ci['low']:+.5f}, {ci['high']:+.5f}] → 채택: {adopted}")
pd.DataFrame([row for rows, _, _ in results.values() for row in rows]).set_index(["지평", "연도"]).round(5)

5거래일: ΔRMSE(보정 − 기준) = +0.00032, 95% 구간 [-0.00014, +0.00079] → 채택: False
20거래일: ΔRMSE(보정 − 기준) = +0.00204, 95% 구간 [+0.00076, +0.00309] → 채택: False


학습 행  평가 행        γ  RMSE 기준  RMSE 뉴스 보정   변화율(%)
지평 연도                                                     
5  2023   246   250 -0.02031  0.04769     0.04735 -0.72394
   2024   496   124 -0.02787  0.04815     0.04844  0.60116
   2025   620   251 -0.02223  0.05960     0.06046  1.44389
   2026   871   184 -0.00409  0.05766     0.05797  0.53089
20 2023   231   250 -0.11427  0.08918     0.09022  1.16169
   2024   481   124 -0.07441  0.09734     0.10119  3.95091
   2025   619   251 -0.03354  0.12182     0.12415  1.91481
   2026   856   169  0.01968  0.13716     0.13893  1.29014

사전 등록한 규칙을 통과하지 못해 채택하지 않는다. 5일은 네 해를 합친 ΔRMSE가 +0.00032(구간 [−0.00014, +0.00079])로 오히려 커졌고 2023년만 좋아졌다(−0.72%). 20일은 네 해 모두 나빠졌고(+1.2 ~ +4.0%) 구간도 0보다 크다. γ는 2026년 20일을 빼면 모두 음수였다(05와 같다). 상승 압력 점수가 높은 날 뒤에 가격이 조금 되돌린 셈이지만, 그 크기가 오차를 줄일 만큼 일정하지 않다. 2024년 평가 행이 절반(124행)인 것은 WTI 공백 때문에 03b의 행 규칙에서 하반기가 빠졌기 때문이다.

In [4]:
# 설명용: 2022–2026년 뉴스 점수와 실제 수익률, 기준 예측의 잔차의 상관
pd.DataFrame({h: {"점수와 실제 수익률": base[h].query("year >= 2022")[["s", "y"]].corr().iloc[0, 1],
                  "점수와 기준 예측의 잔차": base[h].query("year >= 2022").assign(e=lambda f: f["y"] - f["pred"])[["s", "e"]].corr().iloc[0, 1],
                  "기사 있는 날 비율": (base[h].query("year >= 2022")["s"] != 0).mean()} for h in (5, 20)}).round(3)

,5,20
점수와 실제 수익률,0.022,-0.021
점수와 기준 예측의 잔차,0.029,-0.013
기사 있는 날 비율,0.691,0.686


2022–2026년 뉴스 점수와 실제 수익률의 상관은 5일 0.02, 20일 −0.02이고, 기준 예측의 잔차와도 0.03, −0.01이다. 기사가 있는 날은 69%다. 점수는 기사가 나온 날의 움직임을 설명할 뿐(05), 앞으로의 움직임과는 관련이 없다.

In [5]:
common.save_result("news_correction", {
    "eval_years": list(EVAL_YEARS),
    "horizons": {str(h): {"years": rows, "pooled": ci, "adopted": adopted} for h, (rows, ci, adopted) in results.items()},
    "shrink": shrink_table.round(4).to_dict("records"),
})

## 정리

- 서비스 예측은 가격 수준으로 보면 Naive에 가깝다. 평균 예측 크기는 0.2–0.9%로 실제 움직임 크기의 2–16%다. 수익률 RMSE에서 얻을 수 있는 이득은 상관 ρ가 정하고, 5·60일은 사실상 없다.
- 뉴스 보정은 5·20일 모두 사전 등록한 규칙을 통과하지 못해 서비스에 넣지 않는다. 20일은 확실히 나빠졌다.
- live 점수는 매일 쌓이고 있으므로 05와 같은 규칙으로 다시 검증한다(개발 기록 11).